# SGG-Benchmark — End-to-End Tutorial

**Paper:** [REACT++: Efficient Cross-Attention for Real-Time Scene Graph Generation](https://arxiv.org/abs/2603.06386)  
**Paper:** [REACT: Real-time Efficiency and Accuracy Compromise for Tradeoffs in Scene Graph Generation](https://arxiv.org/abs/2405.16116)

This notebook is a complete guide covering every step of the SGG-Benchmark workflow:

| # | Section | What you will do |
|---|---------|-----------------|
| 1 | [Installation](#1-installation) | Set up the Python environment |
| 2 | [Dataset Preparation](#2-dataset-preparation) | Download annotations & images |
| 3 | [Training](#3-training) | Train REACT++ from scratch (SGDet) |
| 4 | [Evaluation (PyTorch)](#4-evaluation-pytorch) | Evaluate a checkpoint with R@K / mR@K |
| 5 | [ONNX Export](#5-onnx-export) | Export to a single deployable `.onnx` file |
| 6 | [ONNX Evaluation](#6-onnx-evaluation) | Run the full test-set eval with ONNX Runtime |
| 7 | [Visualise on Custom Images](#7-visualise-on-custom-images) | Detect scene graphs on your own photos |
| 8 | [Webcam / Video Demo](#8-webcam--video-demo) | Real-time SGG from a webcam or video file |

---
> **Prerequisites**  
> - Linux / macOS with an NVIDIA GPU (CUDA 11.8 or 12.x recommended for GPU inference)  
> - Python 3.11+ (recommended)  
> - `git`, `wget` or `curl` available in your shell

<a id="1-installation"></a>
## 1 — Installation

The project ships a convenience installer script (`scripts/install_uv.sh`) that uses [uv](https://github.com/astral-sh/uv) to create a fully-isolated `.venv` with all dependencies.

### 1.1 Clone the repository

```bash
git clone https://github.com/Maelic/SGG-Benchmark.git
cd SGG-Benchmark
```

### 1.2 Run the installer

```bash
chmod +x scripts/install_uv.sh
./scripts/install_uv.sh
```

The script will:
1. Install `uv` if it is not already present.
2. Create `.venv/` with **Python 3.11**.
3. Install `torch ≥ 2.0` + `torchvision` (GPU wheel automatically selected).
4. Install all pure-Python deps from `requirements.txt`.
5. Install Ultralytics, CLIP, ONNX tooling, and attempt `onnxruntime-gpu`.

### 1.3 Activate the environment

```bash
source .venv/bin/activate
```

### 1.4 Install the package in editable mode

```bash
pip install -e .
```

### 1.5 (Optional) Pin to a specific CUDA build

```bash
# Example — CUDA 12.1
pip install --index-url https://download.pytorch.org/whl/cu121 \
    "torch>=2.0" "torchvision>=0.15"
```

### 1.6 Verify

```python
import torch
print(torch.__version__, "| CUDA available:", torch.cuda.is_available())

import onnxruntime as ort
print("OnnxRuntime:", ort.__version__, "| Providers:", ort.get_available_providers())
```

In [ ]:
# Quick sanity check — run this cell after activating the environment
import torch
import onnxruntime as ort

print(f"PyTorch  : {torch.__version__}")
print(f"CUDA OK  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU      : {torch.cuda.get_device_name(0)}")
print(f"OnnxRuntime: {ort.__version__}")
print(f"Providers: {ort.get_available_providers()}")

<a id="2-dataset-preparation"></a>
## 2 — Dataset Preparation

Three datasets are supported out of the box.  All are published on Hugging Face in a unified COCO-JSON format.

| Dataset  | HF repo | Objects | Predicates | Train | Val | Test |
|----------|---------|--------:|----------:|------:|----:|-----:|
| **VG-150** | [maelic/VG150-coco-format](https://huggingface.co/datasets/maelic/VG150-coco-format) | 150 | 50 | 73 538 | 4 844 | 27 032 |
| **IndoorVG** | [maelic/IndoorVG-coco-format](https://huggingface.co/datasets/maelic/IndoorVG-coco-format) | 84 | 37 | 9 538 | 733 | 4 403 |
| **PSG** | [maelic/PSG-coco-format](https://huggingface.co/datasets/maelic/PSG-coco-format) | 133 | 57 | 45 564 | 1 000 | 2 177 |

### 2.1 Download annotation JSONs

```bash
# Pick one (or all) of: PSG  VG150  IndoorVG
python tools/download_from_hub.py --dataset PSG --save-images
python tools/download_from_hub.py --dataset VG150 --save-images
python tools/download_from_hub.py --dataset IndoorVG --save-images
```

This writes `_annotations.coco.json` files and images under the default paths:

```
datasets/PSG/coco_format/{train,val,test}/_annotations.coco.json
datasets/VG150/VG150_coco_format/{train,val,test}/_annotations.coco.json
datasets/IndoorVG/IndoorVG_coco_format/{train,val,test}/_annotations.coco.json
```

### 2.2 Verify the dataset

```bash
python - <<'EOF'
import json, pathlib
split = "test"
ann_file = pathlib.Path(f"datasets/PSG/coco_format/{split}/_annotations.coco.json")
ann = json.load(open(ann_file))
print(f"Images   : {len(ann['images'])}")
print(f"Relations: {len(ann['rel_annotations'])}")
print(f"Categories: {[c['name'] for c in ann['categories'][:5]]} ...")
EOF
```

In [6]:
import json, pathlib

# Change this to the dataset you downloaded
DATASET   = "VG150"                              # "PSG" | "VG150" | "IndoorVG"
DATA_DIRS = {
    "PSG":      "datasets/PSG/coco_format",
    "VG150":    "datasets/VG150/VG150_coco_format",
    "IndoorVG": "datasets/IndoorVG/IndoorVG_coco_format",
}

data_dir = pathlib.Path(DATA_DIRS[DATASET])

for split in ("train", "val", "test"):
    ann_file = data_dir / split / "_annotations.coco.json"
    if not ann_file.exists():
        print(f"[{split}] NOT FOUND — run: python tools/download_from_hub.py --dataset {DATASET}")
        continue
    ann = json.load(open(ann_file))
    n_imgs = len(ann["images"])
    n_rels = len(ann.get("rel_annotations", []))
    n_cats = len(ann.get("categories", []))
    n_pred = len(ann.get("rel_categories", []))
    print(f"[{split:5s}]  images={n_imgs:6d}  relations={n_rels:7d}  "
          f"obj_classes={n_cats}  pred_classes={n_pred}")

[train]  images= 73538  relations= 439063  obj_classes=150  pred_classes=50
[val  ]  images=  4844  relations=  30133  obj_classes=150  pred_classes=50
[test ]  images= 27032  relations= 153509  obj_classes=150  pred_classes=50


<a id="3-training"></a>
## 3 — Training

The recommended training entry point is `tools/relation_train_net_hydra.py`, driven by YAML configs located in `configs/hydra/`.

### 3.1 Config system overview

```
configs/hydra/
├── default.yaml           ← dataset catalog + shared defaults
├── PSG/
│   ├── REACT++.yaml       ← REACT++ on PSG  ← used in this tutorial
│   ├── REACT.yaml
│   ├── MotifPredictor.yaml
│   └── ...
├── VG/
│   └── REACT++.yaml
└── IndoorVG/
    └── REACT++.yaml
```

Each config file is self-contained: it defines the backbone, relation head, dataset paths, optimizer, and scheduler.  Any field can be overridden on the command line using dot-notation (e.g. `solver.base_lr=0.0005`).

### 3.2 Download a pre-trained backbone (required)

REACT++ uses a YOLO detector pre-trained on the target dataset for object proposals.  Download the matching backbone from the [MODEL ZOO](MODEL_ZOO.md) and place it in `checkpoints/BACKBONES/`.

```bash
mkdir -p checkpoints/BACKBONES
# PSG — YOLOv12m (used by REACT++)
# Download PSG_yolo12m_backbone.pt from MODEL_ZOO.md and save to:
# checkpoints/BACKBONES/PSG_yolo12m_backbone.pt
```

Then update (or override) `pretrained_detector_ckpt` in your config:

```yaml
# configs/hydra/PSG/REACT++.yaml (excerpt)
model:
  pretrained_detector_ckpt: "checkpoints/BACKBONES/PSG_yolo12m_backbone.pt"
```

Or override it directly on the command line (see §3.3).

### 3.3 Launch training — PSG / REACT++ (recommended)

```bash
python tools/relation_train_net_hydra.py \
    --config-name PSG/REACT++ \
    --task sgdet \
    --save-best
```

`--save-best` keeps only the epoch with the best validation `mR@20` instead of every checkpoint.

#### Override any hyperparameter at launch time

```bash
python tools/relation_train_net_hydra.py \
    --config-name PSG/REACT++ \
    --task sgdet \
    --save-best \
    solver.base_lr=0.0005 \
    solver.max_epoch=20 \
    model.pretrained_detector_ckpt=checkpoints/BACKBONES/PSG_yolo12m_backbone.pt \
    output_dir=checkpoints/PSG/my_experiment
```

#### Resume from a checkpoint

```bash
python tools/relation_train_net_hydra.py \
    --config-name PSG/REACT++ \
    --task sgdet \
    --save-best \
    --resume checkpoints/PSG/react_pp/best_model_epoch_5.pth
```

### 3.4 Multi-GPU training

```bash
python -m torch.distributed.launch \
    --master_port 10025 \
    --nproc_per_node=2 \
    tools/relation_train_net_hydra.py \
    --config-name PSG/REACT++ \
    --task sgdet \
    --save-best
```

### 3.5 Training other datasets / models

For datasets:

| Dataset | Config | Notes |
|---------|--------|-------|
| VG-150  | `VG/REACT++` | Change backbone ckpt accordingly |
| IndoorVG | `IndoorVG/REACT++` | Change backbone ckpt accordingly |

For models:

| Model | Config | Notes |
|---------|--------|-------|
| Motif | `model.roi_relation_head.predictor=MotifPredictor` | Classic Neural-Motifs |
| PE-NET | `model.roi_relation_head.predictor=PrototypeEmbeddingNetwork` | Prototype Embedding Network |
| VCTree | `model.roi_relation_head.predictor=VCTreePredictor` | VCTree Predictor |

Be careful! Some predictors require specific configurations like dedicated `lr` or `mlp_head_dim` to reproduce authors results.

### 3.6 Monitor with Weights & Biases

```bash
# Install wandb if needed
pip install wandb
wandb login

# Then simply add to your config or command line:
python tools/relation_train_net_hydra.py \
    --config-name PSG/REACT++ \
    --task sgdet \
    --save-best \
    wandb.use=true \
    wandb.project=sgg-psg
```

In [ ]:
import subprocess, sys

# Preview the REACT++ PSG config (first 80 lines)
result = subprocess.run(
    ["head", "-80", "configs/hydra/PSG/REACT++.yaml"],
    capture_output=True, text=True
)
print(result.stdout)

<a id="4-evaluation-pytorch"></a>
## 4 — Evaluation (PyTorch)

After training (or after downloading a pre-trained checkpoint from the [MODEL ZOO](MODEL_ZOO.md)), evaluate on the test set with `tools/relation_eval_hydra.py`.

The script auto-detects the best checkpoint inside a run directory.

### 4.1 Basic evaluation

```bash
python tools/relation_eval_hydra.py \
    --run-dir checkpoints/PSG/react++_yolo12m \
    --task sgdet
```

### 4.2 Evaluate a specific checkpoint file

```bash
python tools/relation_eval_hydra.py \
    --run-dir checkpoints/PSG/react++_yolo12m \
    --task sgdet \
    --checkpoint best_model_epoch_9.pth
```

### 4.3 Evaluate on a different split

```bash
python tools/relation_eval_hydra.py \
    --run-dir checkpoints/PSG/react++_yolo12m \
    --task sgdet \
    --test-split psg_coco_val
```

### 4.4 Understanding the metrics

| Metric | Description |
|--------|-------------|
| **R@K** | Recall@K — fraction of GT triplets retrieved in the top-K predictions |
| **mR@K** | Mean Recall@K — R@K averaged per predicate class (combats dataset bias) |
| **zR@K** | Zero-shot Recall — R@K restricted to triplets **unseen** during training |
| **F1@K** | Harmonic mean of R@K and mR@K |
| **mAP@50** | Standard COCO mAP for the object detection branch |

> **Rule of thumb:** mR@K is the most informative metric because it is insensitive to the dominance of frequent predicates (e.g. *"on"*, *"of"*).

### 4.5 Load a checkpoint in Python

```python
from pathlib import Path
from omegaconf import OmegaConf
from sgg_benchmark.config.hydra_config import get_cfg, load_config_from_file
from sgg_benchmark.modeling.detector import build_detection_model
from sgg_benchmark.utils.checkpoint import DetectronCheckpointer

run_dir   = Path("checkpoints/PSG/react++_yolo12m")
cfg       = load_config_from_file(run_dir / "config.yml")

model     = build_detection_model(cfg)
model.eval().cuda()

checkpointer = DetectronCheckpointer(cfg, model, save_dir=str(run_dir))
checkpointer.load(str(run_dir / "best_model.pth"))
print("Model loaded — parameters:", sum(p.numel() for p in model.parameters()) / 1e6, "M")
```

In [ ]:
from pathlib import Path
from omegaconf import OmegaConf, open_dict
from sgg_benchmark.config.hydra_config import get_cfg
from sgg_benchmark.modeling.detector import build_detection_model
from sgg_benchmark.utils.checkpoint import DetectronCheckpointer

# ── Point this to your checkpoint directory ──────────────────────────────────
RUN_DIR = Path("checkpoints/PSG/react++_yolo12m")

# Find config file
for candidate in ("config.yml", "config.yaml", "hydra_config.yaml"):
    cfg_file = RUN_DIR / candidate
    if cfg_file.exists():
        break
else:
    raise FileNotFoundError(f"No config found in {RUN_DIR}")

yaml_cfg = OmegaConf.load(cfg_file)
cfg = get_cfg(yaml_cfg)

model = build_detection_model(cfg)
model.eval()

total_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"Model built successfully")
print(f"Total parameters : {total_params:.1f} M")
print(f"Architecture     : {cfg.model.meta_architecture}")
print(f"Backbone         : {cfg.model.backbone.type} ({cfg.model.yolo.size})")

<a id="5-onnx-export"></a>
## 5 — ONNX Export

Exporting to ONNX produces a single, self-contained model file (`model.onnx`) that can run at ~2× the speed of the PyTorch version using ONNX Runtime with CUDA.

### 5.1 Export command

```bash
python tools/export_onnx.py \
    --run-dir checkpoints/PSG/react++_yolo12m \
    --device cuda
```

The script will:
1. Load the best checkpoint from `--run-dir`.
2. Grab a representative input from the dataset.
3. Export to `<run-dir>/model.onnx` using the legacy TorchScript-based ONNX exporter (opset 17).
4. Simplify the graph with `onnx-simplifier`.

### 5.2 Options

| Argument | Default | Description |
|----------|---------|-------------|
| `--run-dir` | *required* | Checkpoint folder |
| `--onnx-path` | `<run-dir>/model.onnx` | Custom output path |
| `--device` | `cuda` | `cuda` or `cpu` for the export forward pass |
| `--num-warmup` | `10` | Warm-up iterations before timing |
| `--image` | *dataset sample* | Optional single image for the export trace |

### 5.3 ONNX model I/O

| Tensor | Shape | Description |
|--------|-------|-------------|
| **input** | `[1, 3, 640, 640]` | Letterboxed RGB image (batch dim dynamic) |
| **boxes** | `[N, D]` | N detected objects; columns are `[x1, y1, x2, y2, score, class_id, …]` |
| **rels** | `[R, 5]` | R predicted relations; columns are `[sub_idx, obj_idx, rel_class, rel_score, pair_score]` |

### 5.4 Benchmark ONNX vs PyTorch latency

```bash
python tools/evaluate.py \
    --model-dir checkpoints/PSG/react++_yolo12m \
    --provider CUDAExecutionProvider \ # will do ONNX evaluation if a .onnx file is present in the directory
    --num-images 200 \ # number of test images to evaluate (set to 0 for the whole set)
    --compare \ # compare ONNX vs PyTorch results side-by-side
    --latency \ # will print latency stats for both ONNX and PyTorch models
    --skip-eval \ # skip metric evaluation and only benchmark latency
```

In [ ]:
import subprocess
from pathlib import Path

RUN_DIR = Path("checkpoints/PSG/react++_yolo12m")

# Export to ONNX (runs in a subprocess so the notebook kernel is not affected)
cmd = [
    "python", "tools/export_onnx.py",
    "--run-dir", str(RUN_DIR),
    "--device", "cuda",
]
print("Running:", " ".join(cmd))
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
if result.returncode != 0:
    print("[STDERR]", result.stderr[-2000:])

onnx_path = RUN_DIR / "model.onnx"
if onnx_path.exists():
    size_mb = onnx_path.stat().st_size / 1e6
    print(f"\n✓  ONNX model saved to {onnx_path}  ({size_mb:.1f} MB)")
else:
    print("\n✗  ONNX file not found — check the error output above.")

<a id="6-onnx-evaluation"></a>
## 6 — ONNX Evaluation

`tools/evaluate.py` can also runs the **full** SGDet evaluation on the PSG test set using ONNX Runtime and PyTorch baseline.  
This is the fastest way to get reproducible benchmark numbers.

### 6.1 Command

```bash
python tools/evaluate.py \
    --model-dir  checkpoints/PSG/react++_yolo12m \
    --provider CUDAExecutionProvider \
    --compare \
    --latency
```

### 6.2 Options

| Argument | Default | Description |
|----------|---------|-------------|
| `--model-dir` | *required* | Folder with `model.onnx` + `config.yml` |
| `--provider` | `CUDAExecutionProvider` | `CUDAExecutionProvider` or `CPUExecutionProvider` |
| `--top-k` | `100` | Max relations per image |
| `--num-images` | `-1` (all) | Limit to first N images (useful for quick tests) |
| `--output-dir` | `<run-dir>` | Where to write logs and result JSONs |
| `--compare` | False | Whether to run both ONNX and PyTorch evaluation for comparison |
| `--latency` | False | Whether to benchmark latency for both ONNX and PyTorch models |
| `--skip-eval` | False | Whether to skip metric evaluation and only benchmark latency |

### 6.3 Outputs

Results are written to `<run-dir>/inference_onnx/`:

```
inference_onnx/
├── eval_onnx.log             ← full evaluation log
└── onnx_eval_summary.json    ← R@K / mR@K / mAP + latency summary
```

In [ ]:
import subprocess, json
from pathlib import Path

RUN_DIR  = Path("../checkpoints/PSG/react++_yolov8m")

cmd = [
    "python", "tools/evaluate.py",
    "--model-dir",  str(RUN_DIR),
    "--provider", "CUDAExecutionProvider",
    "--num-images", "200",
    "--compare",
    "--latency",
]
print("Running:", " ".join(cmd))
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout[-4000:] if len(result.stdout) > 4000 else result.stdout)
if result.returncode != 0:
    print("[STDERR]", result.stderr[-2000:])

# Pretty-print the saved summary
summary_path = RUN_DIR / "inference_onnx" / "onnx_eval_summary.json"
detailed = json.load(open(summary_path)) if summary_path.exists() else None
metrics = {}
if detailed:
    mr = detailed.get("sgdet_mean_recall", {})
    r  = detailed.get("sgdet_recall", {})
    zs = detailed.get("sgdet_zeroshot_recall", {})
    for k in ("20", "50", "100"):
        if k in mr:
            v = mr[k]
            metrics[f"mR@{k}"] = round(v * 100, 2) if isinstance(v, float) else None
        if k in r:
            v = r[k]
            if isinstance(v, list):
                metrics[f"R@{k}"] = round(sum(v) / len(v) * 100, 2)
# compute F1 scores if we have both R and mR
for k in ("20", "50", "100"):
    r_key = f"R@{k}"
    mr_key = f"mR@{k}"
    f1_key = f"F1@{k}"
    if r_key in metrics and mr_key in metrics:
        r = metrics[r_key]
        mr = metrics[mr_key]
        if r is not None and mr is not None and (r + mr) > 0:
            metrics[f1_key] = round(2 * r * mr / (r + mr), 2)

# get latency from eval_summary.json if available
if detailed:
    lat = detailed.get("mean_latency_ms", {})
    if lat is not None:
        metrics["forward_latency_ms"] = round(float(lat), 1)

    lat_e2e = detailed.get("mean_full_latency_ms", {})
    if lat_e2e is not None:
        metrics["e2e_latency_ms"] = round(float(lat_e2e), 1)

print("\n── ONNX Eval Summary ─────────────────────────────")
if metrics:
    for k, v in metrics.items():
        print(f"  {k}: {v}")


# compare with the PyTorch evaluation summary if it exists
summary_path = RUN_DIR / "inference_sgdet" / "eval_summary.json"
detailed = json.load(open(summary_path)) if summary_path.exists() else None
metrics = {}
if detailed:
    mr = detailed.get("sgdet_mean_recall", {})
    r  = detailed.get("sgdet_recall", {})
    zs = detailed.get("sgdet_zeroshot_recall", {})
    for k in ("20", "50", "100"):
        if k in mr:
            v = mr[k]
            metrics[f"mR@{k}"] = round(v * 100, 2) if isinstance(v, float) else None
        if k in r:
            v = r[k]
            if isinstance(v, list):
                metrics[f"R@{k}"] = round(sum(v) / len(v) * 100, 2)
# compute F1 scores if we have both R and mR
for k in ("20", "50", "100"):
    r_key = f"R@{k}"
    mr_key = f"mR@{k}"
    f1_key = f"F1@{k}"
    if r_key in metrics and mr_key in metrics:
        r = metrics[r_key]
        mr = metrics[mr_key]
        if r is not None and mr is not None and (r + mr) > 0:
            metrics[f1_key] = round(2 * r * mr / (r + mr), 2)

# get latency from eval_summary.json if available
if detailed:
    lat = detailed.get("eval_timing", {}).get("mean_forward_ms")
    if lat is not None:
        metrics["forward_latency_ms"] = round(float(lat), 1)

print("\n── PyTorch Eval Summary ─────────────────────────────")
if metrics:
    for k, v in metrics.items():
        print(f"  {k}: {v}")


── ONNX Eval Summary ─────────────────────────────
  mR@20: 12.05
  R@20: 22.78
  mR@50: 15.42
  R@50: 28.73
  mR@100: 16.51
  R@100: 30.84
  F1@20: 15.76
  F1@50: 20.07
  F1@100: 21.51
  forward_latency_ms: 14.6
  e2e_latency_ms: 17.8

── PyTorch Eval Summary ─────────────────────────────
  mR@20: 12.22
  R@20: 22.89
  mR@50: 16.31
  R@50: 29.96
  mR@100: 18.45
  R@100: 34.09
  F1@20: 15.93
  F1@50: 21.12
  F1@100: 23.94
  forward_latency_ms: 18.7


<a id="7-visualise-on-custom-images"></a>
## 7 — Visualise on Custom Images

The codebase ships two demo helpers:

- **`demo/demo_model.py`** — PyTorch inference wrapper (`SGG_Model`)
- **`demo/onnx_model.py`** — ONNX Runtime inference wrapper (`SGG_ONNX_Model`)

Both share the same visualisation API.  We recommend the ONNX wrapper for speed.

### 7.1 Detect a scene graph on a single image (ONNX)

```python
import cv2
from demo.onnx_model import SGG_ONNX_Model

model = SGG_ONNX_Model(
    onnx_path   = "checkpoints/PSG/react++_yolo12m/react_pp_yolo12m.onnx",
    provider    = "CUDAExecutionProvider",   # or "CPUExecutionProvider"
    rel_conf    = 0.10,   # relation confidence threshold
    box_conf    = 0.40,   # object detection threshold
)

image = cv2.imread("path/to/your/image.jpg")
annotated_img, graph = model.predict(image, visu_type="image")

cv2.imwrite("output.jpg", annotated_img)
```

### 7.2 Detect a scene graph on a single image (PyTorch)

```python
from demo.demo_model import SGG_Model

model = SGG_Model(
    config_path = "checkpoints/PSG/react++_yolo12m/config.yml",
    weight_file = "checkpoints/PSG/react++_yolo12m/best_model.pth",
    rel_conf    = 0.10,
    box_conf    = 0.40,
)

annotated_img, graph = model.predict(image, visu_type="image")
```

<a id="8-webcam--video-demo"></a>
## 8 — Webcam / Video Demo

Two demo scripts are available in the `demo/` folder:

| Script | Model | Notes |
|--------|-------|-------|
| `demo/webcam_demo.py` | PyTorch `.pth` | Full flexibility, slower |
| `demo/webcam_demo_onnx.py` | ONNX Runtime | Recommended — ~2× faster |

### 8.1 PyTorch webcam demo

```bash
cd demo
python webcam_demo.py \
    --config ../checkpoints/PSG/react++_yolo12m/config.yml \
    --weights ../checkpoints/PSG/react++_yolo12m/best_model.pth \
    --rel_conf 0.15 \
    --box_conf 0.40
```

Press **`q`** to quit, **`p`** to pause.

### 8.2 ONNX webcam demo (recommended)

```bash
cd demo
python webcam_demo_onnx.py \
    --onnx   ../checkpoints/PSG/react++_yolo12m/react_pp_yolo12m.onnx \
    --provider CUDAExecutionProvider \
    --rel_conf 0.15 \
    --box_conf 0.40
```

### 8.3 Common options

| Argument | Default | Description |
|----------|---------|-------------|
| `--webcam` | `0` | Camera device index (or a video file path) |
| `--rel_conf` | `0.10` | Relation score threshold |
| `--box_conf` | `0.50` | Object detection score threshold |
| `--tracking` | off | Enable object tracking across frames (requires `pip install boxmot`) |
| `--save_path` | — | Save the annotated output to a video file |
| `--visu_type` | `bbox` | `bbox` draws boxes; `image` also renders the graph structure |

### 8.4 Run on a video file

```bash
python demo/webcam_demo_onnx.py \
    --onnx   ../checkpoints/PSG/react++_yolo12m/react_pp_yolo12m.onnx \
    --webcam  path/to/my_video.mp4 \
    --save_path output_video.avi \
    --rel_conf 0.15
```

### 8.5 Enable object tracking

Object tracking keeps consistent IDs for each detected entity across frames, making the scene graph temporally stable.

```bash
pip install boxmot

python demo/webcam_demo_onnx.py \
    --onnx   ../checkpoints/PSG/react++_yolo12m/react_pp_yolo12m.onnx \
    --tracking
```

In [ ]:
"""
Run the ONNX webcam demo from the notebook.
A new window will open — press 'q' to quit.
This cell does nothing if no camera is available.
"""
import subprocess, shutil

RUN_DIR = "../checkpoints/PSG/react++_yolo12m"

if shutil.which("python") is None:
    print("Python not found on PATH — please run the demo from a terminal.")
else:
    cmd = [
        "python", "../demo/webcam_demo_onnx.py",
        "--onnx",     f"{RUN_DIR}/model.onnx",
        "--provider", "CUDAExecutionProvider",
        "--rel_conf", "0.15",
        "--box_conf", "0.40",
    ]
    print("Launching demo:", " ".join(cmd))
    print("(Press 'q' in the opened window to stop)")
    subprocess.run(cmd)

## Appendix — Frequently Asked Questions

### Q: Can I train on my own dataset?

Yes. Annotate your images with the [SGG-Annotate](https://github.com/Maelic/SGG-Annotate) tool, which exports in the COCO-JSON format this codebase expects.  Then register a new catalog entry in `configs/hydra/default.yaml` and point your training config to it.

### Q: The model predicts only `"on"` and `"of"`, how do I fix this?

This is a class-imbalance issue common in SGG.  Try:
1. Enabling a re-weighting loss — set `loss.loss_type: "bce_reweight"` in your config.
2. Lowering `rel_conf` to expose more diverse predictions.
3. Checking `mR@K` instead of `R@K`: mR@K reveals how the tail classes are doing.

### Q: ONNX export fails with a `SymInt` error

Ensure you are using the **legacy** ONNX exporter (opset ≤ 17) and not the new dynamo-based one.  The `export_onnx.py` script already uses `torch.onnx.utils.export` directly to avoid this.  If you modified the model, make sure no `torch.export`-incompatible ops are introduced.

### Q: How do I add a new relation head?

1. Implement your predictor in `sgg_benchmark/modeling/roi_heads/relation_head/`.
2. Register it in `roi_relation_predictors.py`.
3. Create a Hydra config file under `configs/hydra/<Dataset>/MyPredictor.yaml`.

---

## Useful Links

| Resource | URL |
|----------|-----|
| Paper (REACT++) | https://arxiv.org/abs/2603.06386 |
| Paper (REACT) | https://arxiv.org/abs/2405.16116 |
| GitHub | https://github.com/Maelic/SGG-Benchmark |
| Model Zoo | [docs/MODEL_ZOO.md](MODEL_ZOO.md) |
| Dataset docs | [docs/DATASET.md](DATASET.md) |
| Metrics explained | [docs/METRICS.md](METRICS.md) |
| SGG-Annotate | https://github.com/Maelic/SGG-Annotate |
| Hugging Face | https://huggingface.co/maelic |